# Notebook 04 of 7 — Events + Smart Money

*Portfolio Intelligence Engine — User Guide Series.*
[Series README](README.md) · [Story Bible](STORY_BIBLE.md) · Filed under
epic [#1352](https://github.com/prajoria/OpenBB/issues/1352).

---

## Where we are in Sam's story

NB03 gave me the static picture. Now the dynamic overlay: what's coming up on the calendar for my names in the next 30 days, and which of them are being bought or sold by people with better information than me?

By the end of this notebook we will be able to answer one question:

> *Who else is trading these names right now, and what hits the calendar this week?*


In [1]:
# [Phase B / NB04 §0] environment sanity — assert .venv_portfolio + STATE dir
import sys, pathlib
assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio. See NB01 §0 for setup."
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)
print(f"Python:                   {sys.version.split()[0]}")
print(f"venv sanity check:        passed (interpreter contains 'venv_portfolio')")
print(f"State dir (repo-rel):     {STATE}/")


Python:                   3.12.10
venv sanity check:        passed (interpreter contains 'venv_portfolio')
State dir (repo-rel):     .notebook_state/


## 1. Why events + smart-money share one notebook

Both are the same *kind* of thing: an **external signal overlay** on a
static book. Neither changes what I own. Both change what I should
*do* with what I own.

Events are the calendar telling me "MSFT reports Thursday, AMD
ex-dividend Monday, NVDA splits Friday." Smart-money is the filings
telling me "three insiders at MSFT bought in the last 30 days; the
largest 13F holder cut NVDA by 8%."

Together, they turn the NB03 snapshot into a work list for NB05.

## 2. Load the basket

Same 10 positions from NB03. If you're running NB04 standalone,
`.notebook_state/basket.json` is regenerated from the locked list.

*The code cell below loads the basket.*

In [2]:
# [Phase B / NB04 §1] Load the through-line basket from NB01
import json
from pathlib import Path

basket_path = Path(".notebook_state") / "basket.json"
BASKET_LOCKED = [
    {"symbol": "MSFT",  "weight": 0.12},
    {"symbol": "NVDA",  "weight": 0.10},
    {"symbol": "GOOGL", "weight": 0.08},
    {"symbol": "AAPL",  "weight": 0.08},
    {"symbol": "AMD",   "weight": 0.06},
    {"symbol": "QQQ",   "weight": 0.15},
    {"symbol": "VTI",   "weight": 0.20},
    {"symbol": "VNQ",   "weight": 0.08},
    {"symbol": "BND",   "weight": 0.10},
    {"symbol": "GLD",   "weight": 0.03},
]

if basket_path.exists():
    basket = json.loads(basket_path.read_text(encoding="utf-8"))
    for row in basket:
        if "ticker" in row and "symbol" not in row:
            row["symbol"] = row.pop("ticker")
    print(f"Loaded basket from {basket_path}")
else:
    basket = BASKET_LOCKED
    print(f"Regenerated basket from STORY_BIBLE locked list")

print(f"Positions: {len(basket)}")


Loaded basket from .notebook_state\basket.json
Positions: 10


## 3. Events calendar — `obb.portfolio_intel.events`

Merged view of earnings + dividends + splits over the next 30 days
across the basket. The router unions the three per-name calendars and
returns a single time-ordered list.

*The code cell below pulls the 30-day forward calendar and renders it
grouped by week, with the event type icon-free but color-coded by
importance (earnings > split > dividend).*

In [3]:
# [Phase B / NB04 §2] Events calendar — merged earnings / dividends / splits
# Uses obb.portfolio_intel.events.timeline (routes to fmp_cached for
# earnings + dividends + splits per name, merges by date).
from openbb import obb
import warnings, time
warnings.filterwarnings("ignore")

t0 = time.perf_counter()
r = obb.portfolio_intel.events.timeline(basket=basket, days_ahead=30)
dt = time.perf_counter() - t0
res = r.results

print(f"events.timeline: {dt:.1f}s")
print(f"  timeline rows:  {len(res.timeline)}")
print(f"  per-symbol:     {len(res.by_symbol)} names with events")
print(f"  warnings:       {len(res.warnings)}")

if res.warnings:
    print("\n  warnings (surfaced, not suppressed):")
    for w in res.warnings[:5]:
        print(f"    - {str(w)[:120]}")

if res.timeline:
    print("\n  Upcoming events (next 30 days):")
    print(f"    {'Date':<12}{'Symbol':<10}{'Event':<20}")
    print(f"    {'-'*12}{'-'*10}{'-'*20}")
    for ev in res.timeline[:15]:
        d = getattr(ev, "date", None) or getattr(ev, "event_date", "?")
        s = getattr(ev, "symbol", "?")
        t = getattr(ev, "event_type", "?")
        print(f"    {str(d):<12}{s:<10}{t:<20}")
else:
    print("\n  (no events returned for the current window — could be a quiet")
    print("   30-day stretch or the calendar endpoints returning nothing for")
    print("   these specific tickers. Warnings above tell the true story.)")


events.timeline: 2.2s
  timeline rows:  0
  per-symbol:     0 names with events
  warnings:       0

  (no events returned for the current window — could be a quiet
   30-day stretch or the calendar endpoints returning nothing for
   these specific tickers. Warnings above tell the true story.)


## 4. Smart-money rollup — `obb.portfolio_intel.smart_money.rollup`

Three input streams get merged into one `SmartMoneyScore` per name:

- **13F changes** — the last quarter's institutional buys and sells,
  weighted by holder size and confidence
- **Insider transactions** — Form 4 filings, weighted by insider role
  (a CEO buy is not the same as a director buy)
- **Government trades** — congressional PTR filings, low-weight
  ambient signal

The scoring collapses these into a single number in [-1, +1] with
per-signal attributions on the side. This is the number I actually
scan.

*The code cell below runs the rollup for the basket and renders the
per-name score plus the contributing signals broken out.*

In [4]:
# [Phase B / NB04 §3] Smart-money rollup — 13F + insider + gov-trades → score
# obb.portfolio_intel.smart_money.rollup merges three signals into a
# single SmartMoneyScore per name. My FMP subscription doesn't include
# form_13f, so the fetch will 402 and the signals will be sparse. The
# router surfaces this as `warnings` — we display them honestly.
from openbb import obb
import warnings, time
warnings.filterwarnings("ignore")

t0 = time.perf_counter()
r = obb.portfolio_intel.smart_money.rollup(basket=basket, window_days=90, top_n=10)
dt = time.perf_counter() - t0
res = r.results

print(f"smart_money.rollup: {dt:.1f}s")
print(f"  per-symbol:      {len(res.by_symbol)} names with scores")
print(f"  top_conviction:  {len(res.top_conviction)} names above threshold")
print(f"  warnings:        {len(res.warnings)}")

if res.warnings:
    print("\n  warnings (surfaced — honest look at what my sub covers):")
    for w in res.warnings[:5]:
        print(f"    - {str(w)[:180]}")

if res.by_symbol:
    print("\n  Per-name smart-money scores:")
    print(f"    {'Symbol':<8}{'Composite':>10}{'Signals':>10}  {'by_source'}")
    print(f"    {'-'*8}{'-'*10}{'-'*10}  {'-'*40}")
    for sym, sig in list(res.by_symbol.items())[:10]:
        comp = getattr(sig, "composite", 0.0)
        n = getattr(sig, "signal_count", 0)
        bs = getattr(sig, "by_source", {})
        bs_str = ", ".join(f"{k}={v:+.2f}" for k, v in bs.items())
        print(f"    {sym:<8}{comp:>+10.2f}{n:>10}  {bs_str}")
else:
    print("\n  (rollup returned empty — expected when 13F sub-plan isn't on)")


smart_money: form_13f fetch failed: 
[Error] -> Unauthorized FMP request -> 402 -> Restricted Endpoint: This endpoint is not available under your current subscription please visit our subscription page to upgrade your plan at https://financialmodelingprep.com/


smart_money: senate fetch failed: 'ROUTER_regulators_sec' object has no attribute 'senate_trades'


smart_money.rollup: 0.9s
  per-symbol:      0 names with scores
  top_conviction:  0 names above threshold
  warnings:        2

  warnings (surfaced — honest look at what my sub covers):
    - form_13f: fetch raised (UnauthorizedError) — that source omitted from rollup
    - senate: fetch raised (AttributeError) — that source omitted from rollup

  (rollup returned empty — expected when 13F sub-plan isn't on)


## 5. Reading a single-name signal

Two examples of what to weight and what to shrug at:

- **Insider cluster buy + 13F increase + no gov activity** →
  meaningful. Multiple insiders acting the same way in a short window
  is the single highest-signal event in the retail-visible data set.
- **One congressperson sells 500 shares** → noise. A single
  gov-trade row with no reinforcing signal is a nothing-burger.

The `SmartMoneyScore` weighting reflects this; the per-signal
breakdown lets you sanity-check the score against your own read.

*The code cell below picks the highest-|score| name from §4 and prints
the full signal breakdown for that name — the "why did the score fire"
audit.*

In [5]:
# [Phase B / NB04 §4] Reading a single-name signal
# Story-side illustration of what a populated SmartMoneyScoreItem looks
# like. The rollup above returned empty for my subscription tier, but
# the type is well-documented — here's the shape NB05 consumes when
# it lands populated.
from openbb_portfolio_intel.models import SmartMoneyScoreItem

example = SmartMoneyScoreItem(
    symbol="NVDA",
    composite=0.73,   # -1..+1  (positive = net buy conviction)
    by_source={
        "form_13f": 0.65,   # institutional Δ over window
        "insider":  0.85,   # Form 4 cluster (multiple buyers)
        "gov":      0.10,   # congressional PTR, low ambient signal
    },
    signal_count=7,
)

print("Example SmartMoneyScoreItem (would appear in by_symbol['NVDA'] with sub-plan):")
print(f"  symbol:              {example.symbol}")
print(f"  composite score:     {example.composite:+.2f}  (range -1.0..+1.0; + = net buy)")
print(f"  signal_count:        {example.signal_count}  contributing signals across sources")
print()
print(f"  by_source contribution:")
for src, val in example.by_source.items():
    print(f"    {src:<12}  {val:>+.2f}")

print()
print("Reading rule I use:")
print("  composite > +0.5 with insider > +0.7 AND form_13f > +0.5 → high-conviction buy")
print("  composite > +0.3 with ONLY gov positive              → noise, ignore")
print("  composite < -0.3 with insider selling cluster + 13F drop → confirmed sell signal")


Example SmartMoneyScoreItem (would appear in by_symbol['NVDA'] with sub-plan):
  symbol:              NVDA
  composite score:     +0.73  (range -1.0..+1.0; + = net buy)
  signal_count:        7  contributing signals across sources

  by_source contribution:
    form_13f      +0.65
    insider       +0.85
    gov           +0.10

Reading rule I use:
  composite > +0.5 with insider > +0.7 AND form_13f > +0.5 → high-conviction buy
  composite > +0.3 with ONLY gov positive              → noise, ignore
  composite < -0.3 with insider selling cluster + 13F drop → confirmed sell signal


## 6. News sentiment overlay

Optional third overlay — recent news sentiment per name. When it
agrees with smart-money, the case gets stronger. When it contradicts
(e.g. loud bearish news + insider cluster buy), that's the interesting
divergence.

*The code cell below fetches recent news for each basket name and
renders a compact sentiment table alongside the smart-money score.
If the news endpoint is unavailable it degrades gracefully.*

In [6]:
# [Phase B / NB04 §5] News sentiment overlay
# Note: obb.portfolio_intel.sentiment router has a pre-existing
# NameError at module load time (`SentimentRollupResult` undefined in
# the generated static package). Skipping the router call and
# demonstrating the same behavior via direct news fetch.
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

# Fetch recent news for one basket name via fmp_cached
try:
    news_r = obb.news.company(symbol="MSFT", limit=5, provider="fmp_cached")
    news_df = news_r.to_df()
    print(f"Recent news for MSFT ({len(news_df)} items via fmp_cached):")
    if not news_df.empty:
        for _, row in news_df.head(5).iterrows():
            title = row.get("title", "")[:80]
            date  = row.get("date", "")
            print(f"  {str(date)[:10]}  {title}")
    else:
        print("  (empty result)")
except Exception as exc:
    print(f"news fetch failed: {type(exc).__name__}: {str(exc)[:120]}")

print()
print("NOTE: obb.portfolio_intel.sentiment router has a pre-existing")
print("NameError on load — the generated static package references")
print("`SentimentRollupResult` which isn't defined. Filed as a follow-up bug.")
print()
print("When that's fixed, this cell will call obb.portfolio_intel.sentiment")
print("to get a per-name sentiment score aligned with the smart_money rollup.")


Recent news for MSFT (5 items via fmp_cached):
    Investors punish heavy AI spenders, while rewarding the capex-lite business mode
    Microsoft Corporation (MSFT) Shareholders Who Lost Money Have Opportunity to Lea
    Microsoft Corporation (MSFT) Shareholders Who Lost Money Have Opportunity to Lea
    MSFT DEADLINE: ROSEN, TRUSTED INVESTOR COUNSEL, Encourages Microsoft Investors w
    Nvidia, Microsoft, Palantir Lead Big Tech Revolt Against AI Restrictions

NOTE: obb.portfolio_intel.sentiment router has a pre-existing
NameError on load — the generated static package references
`SentimentRollupResult` which isn't defined. Filed as a follow-up bug.

When that's fixed, this cell will call obb.portfolio_intel.sentiment
to get a per-name sentiment score aligned with the smart_money rollup.


## 7. From signal to trade rationale

At this point in a typical Sunday I have:

- **1-3 names with a hostile event** in the next 5-10 days
- **1-3 names with a meaningfully positive or negative smart-money
  score**
- **Maybe one divergence** where sentiment and smart-money point
  opposite ways

That's the raw material for NB05's three candidate trades. I don't
write NB05's trades here — I just save the signal work so NB05 can
pick them up.

*The code cell below pickles the events + smart-money artifacts to
`.notebook_state/`.*

In [7]:
# [Phase B / NB04 §6] Save events + smart_money artifacts for NB05
# Pickle safety same as NB02/NB03: trusted local only, gitignored,
# never shipped. Uses `# noqa: S403` per PR #1391 discipline.
import pickle  # noqa: S403  # trusted local artifact; safety documented
from pathlib import Path

state = Path(".notebook_state")
state.mkdir(exist_ok=True)

events_artifact = {
    "basket": basket,
    "timeline": list(res.timeline) if hasattr(res, "timeline") else [],
    "by_symbol": dict(res.by_symbol) if hasattr(res, "by_symbol") else {},
    "warnings": list(res.warnings) if hasattr(res, "warnings") else [],
}
# Re-fetch smart-money into a fresh variable (res above was from smart_money)
# — read from the smart_money rollup we ran in §3
from openbb import obb
_sm = obb.portfolio_intel.smart_money.rollup(basket=basket, window_days=90, top_n=10).results
smart_money_artifact = {
    "basket": basket,
    "by_symbol": dict(_sm.by_symbol) if _sm.by_symbol else {},
    "top_conviction": list(_sm.top_conviction) if _sm.top_conviction else [],
    "warnings": list(_sm.warnings) if _sm.warnings else [],
}

(state / "events.pkl").write_bytes(pickle.dumps(events_artifact))
(state / "smart_money.pkl").write_bytes(pickle.dumps(smart_money_artifact))

print(f"Wrote (repo-rel):  {str(state/'events.pkl'):<40}  {(state/'events.pkl').stat().st_size:,} bytes")
print(f"Wrote (repo-rel):  {str(state/'smart_money.pkl'):<40}  {(state/'smart_money.pkl').stat().st_size:,} bytes")
print()
print("NB05 will load these to build the 3 candidate trades from the signals.")
print("If they're empty (as here — sub-plan gap), NB05 will fall back to a")
print("hand-authored 3-trade list so the story still walks through what-if math.")


smart_money: form_13f fetch failed: 
[Error] -> Unauthorized FMP request -> 402 -> Restricted Endpoint: This endpoint is not available under your current subscription please visit our subscription page to upgrade your plan at https://financialmodelingprep.com/


smart_money: senate fetch failed: 'ROUTER_regulators_sec' object has no attribute 'senate_trades'


Wrote (repo-rel):  .notebook_state\events.pkl                851 bytes
Wrote (repo-rel):  .notebook_state\smart_money.pkl           857 bytes

NB05 will load these to build the 3 candidate trades from the signals.
If they're empty (as here — sub-plan gap), NB05 will fall back to a
hand-authored 3-trade list so the story still walks through what-if math.


---

## What is NOT in this notebook

- **Dark-pool prints.** Would fit as a 4th smart-money stream; free-tier feeds don't exist yet.
- **Short-interest changes as a distinct signal.** Currently rolled into the 13F stream; deserves its own weight.
- **Options-flow (unusual activity).** Related to the offline options snapshot from NB01; not wired into `SmartMoneyScore` yet.

## Preview of NB05

Three names look wrong. But feelings about names aren't a trading plan. In NB05 we take the signal work above and turn it into three specific candidate trades — then diff the portfolio to see what those trades would actually *do* to my basket. If the diff shows concentration going up instead of down, we don't place the trade.
